# Lab 15 — Activation, Initialisation and L2 Regularisation
Breast Cancer Wisconsin teaching comparison. Streamlined Colab edition.

In [ ]:
import numpy as np,pandas as pd,torch,torch.nn as nn,torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,f1_score
X,y=load_breast_cancer(return_X_y=True); y=(y==0).astype(int); Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y); sc=StandardScaler(); Xtr=sc.fit_transform(Xtr).astype('float32'); Xte=sc.transform(Xte).astype('float32'); tr=DataLoader(TensorDataset(torch.tensor(Xtr),torch.tensor(ytr)),batch_size=32,shuffle=True); device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class MLP(nn.Module):
 def __init__(self,act='relu',init='he'):
  super().__init__(); A=nn.ReLU if act=='relu' else nn.Sigmoid; self.net=nn.Sequential(nn.Linear(30,64),A(),nn.Linear(64,32),A(),nn.Linear(32,2));
  for m in self.modules():
   if isinstance(m,nn.Linear):
    nn.init.kaiming_normal_(m.weight,nonlinearity='relu') if init=='he' else nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
 def forward(self,x): return self.net(x)

In [ ]:
def run(act,init,l2):
 model=MLP(act,init).to(device); opt=optim.Adam(model.parameters(),lr=1e-3,weight_decay=l2); loss_fn=nn.CrossEntropyLoss()
 for e in range(60):
  for x,y in tr:
   x=x.to(device); y=y.to(device); opt.zero_grad(); loss=loss_fn(model(x),y); loss.backward(); opt.step()
 with torch.no_grad(): pred=model(torch.tensor(Xte).to(device)).argmax(1).cpu().numpy()
 return accuracy_score(yte,pred),f1_score(yte,pred)
rows=[]
for cfg in [('relu','he',0),('relu','xavier',0),('sigmoid','xavier',0),('sigmoid','he',0),('relu','he',1e-4),('relu','he',1e-3),('relu','he',1e-2)]: rows.append([*cfg,*run(*cfg)])
print(pd.DataFrame(rows,columns=['Activation','Initialisation','L2','Accuracy','F1']))